In [1]:
import json
from pathlib import Path

import pandas as pd

In [2]:
caminho_dados = Path("../dados/dados_nivel_2.json")

print("Arquivo encontrado:", caminho_dados.exists())
print("Diretório atual:", Path.cwd())

Arquivo encontrado: True
Diretório atual: c:\Users\maria\Desktop\desafio-estagio-ia\nivel_2


In [3]:
with open(caminho_dados, "r", encoding="utf-8") as arquivo:
    dados_nivel_2 = json.load(arquivo)

taxa_cambio_usd_brl = float(
    dados_nivel_2["taxa_cambio_usd_brl"]
)

df_nivel_2 = pd.DataFrame(
    dados_nivel_2["operacoes"]
)

df_nivel_2.head()

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-00133,CLI-014,2026-03-06,23640.97,BRL,pix,pagamento,Mirante Transportes ME,
1,OP-00103,CLI-011,2026-03-04,9447.52,BRL,cartao,saque,Quartzo Industria SA,
2,OP-00223,CLI-023,2026-04-13,2891.48,BRL,ted,transferencia_enviada,Gama Importacao SA,
3,OP-00265,CLI-028,2026-04-23,5636.46,BRL,pix,saque,Nauta Atacado ME,
4,OP-00099,CLI-010,2026-03-20,6641.24,BRL,ted,pagamento,Delta Trading LTDA,


In [4]:
print("Quantidade inicial de registros:", len(df_nivel_2))
print("Clientes únicos:", df_nivel_2["cliente_id"].nunique())
print("Duplicidades integrais:", df_nivel_2.duplicated().sum())

display(df_nivel_2.isna().sum())

Quantidade inicial de registros: 322
Clientes únicos: 30
Duplicidades integrais: 5


id             0
cliente_id     0
data           7
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

In [5]:
for coluna in ["moeda", "canal", "tipo"]:
    print(f"\nValores encontrados em {coluna}:")
    print(df_nivel_2[coluna].value_counts(dropna=False))


Valores encontrados em moeda:
moeda
BRL    315
USD      7
Name: count, dtype: int64

Valores encontrados em canal:
canal
ted        82
especie    67
pix        61
cartao     58
boleto     54
Name: count, dtype: int64

Valores encontrados em tipo:
tipo
deposito                  69
pagamento                 68
transferencia_enviada     68
transferencia_recebida    60
saque                     57
Name: count, dtype: int64


In [6]:
df_nivel_2 = df_nivel_2.drop_duplicates().copy()

df_nivel_2["data"] = pd.to_datetime(
    df_nivel_2["data"],
    errors="coerce"
)

df_nivel_2["data_ausente"] = (
    df_nivel_2["data"].isna()
)

In [7]:
for coluna in ["canal", "tipo"]:
    df_nivel_2[coluna] = (
        df_nivel_2[coluna]
        .astype("string")
        .str.strip()
        .str.lower()
    )

df_nivel_2["moeda"] = (
    df_nivel_2["moeda"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [8]:
df_nivel_2["valor"] = pd.to_numeric(
    df_nivel_2["valor"],
    errors="coerce"
)

print(
    "Valores que não puderam ser convertidos:",
    df_nivel_2["valor"].isna().sum()
)

Valores que não puderam ser convertidos: 0


In [9]:
df_nivel_2["valor_brl"] = df_nivel_2["valor"]

mascara_usd = df_nivel_2["moeda"] == "USD"

df_nivel_2.loc[mascara_usd, "valor_brl"] = (
    df_nivel_2.loc[mascara_usd, "valor"]
    * taxa_cambio_usd_brl
)

df_nivel_2.head()

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,data_ausente,valor_brl
0,OP-00133,CLI-014,2026-03-06,23640.97,BRL,pix,pagamento,Mirante Transportes ME,,False,23640.97
1,OP-00103,CLI-011,2026-03-04,9447.52,BRL,cartao,saque,Quartzo Industria SA,,False,9447.52
2,OP-00223,CLI-023,2026-04-13,2891.48,BRL,ted,transferencia_enviada,Gama Importacao SA,,False,2891.48
3,OP-00265,CLI-028,2026-04-23,5636.46,BRL,pix,saque,Nauta Atacado ME,,False,5636.46
4,OP-00099,CLI-010,2026-03-20,6641.24,BRL,ted,pagamento,Delta Trading LTDA,,False,6641.24


In [10]:
print("Registros após a limpeza:", len(df_nivel_2))
print("Clientes únicos:", df_nivel_2["cliente_id"].nunique())
print("Duplicidades restantes:", df_nivel_2.duplicated().sum())
print("Datas ausentes ou inválidas:", df_nivel_2["data"].isna().sum())
print("Valores BRL ausentes:", df_nivel_2["valor_brl"].isna().sum())
print("Moedas encontradas:", df_nivel_2["moeda"].unique())

Registros após a limpeza: 317
Clientes únicos: 30
Duplicidades restantes: 0
Datas ausentes ou inválidas: 6
Valores BRL ausentes: 0
Moedas encontradas: <StringArray>
['BRL', 'USD']
Length: 2, dtype: string


In [28]:
df_regra_1_n2 = df_nivel_2[
    df_nivel_2["data"].notna()
].copy()

df_regra_1_n2["dia"] = (
    df_regra_1_n2["data"].dt.normalize()
)

In [12]:
resumo_regra_1_n2 = (
    df_regra_1_n2
    .groupby(
        ["cliente_id", "dia"],
        as_index=False
    )
    .agg(
        quantidade_operacoes=("id", "count"),
        soma_operacoes_brl=("valor_brl", "sum"),
        maior_operacao_brl=("valor_brl", "max")
    )
)

resumo_regra_1_n2.head()

,cliente_id,dia,quantidade_operacoes,soma_operacoes_brl,maior_operacao_brl
0,CLI-001,2026-03-06,1,668.12,668.12
1,CLI-001,2026-03-12,1,1605.71,1605.71
2,CLI-001,2026-03-15,1,25110.15,25110.15
3,CLI-001,2026-05-13,1,535.91,535.91
4,CLI-001,2026-05-14,1,2060.47,2060.47


In [13]:
resumo_regra_1_n2["sinalizado_regra_1"] = (
    resumo_regra_1_n2["quantidade_operacoes"].ge(3)
    & resumo_regra_1_n2["soma_operacoes_brl"].gt(50_000)
    & resumo_regra_1_n2["maior_operacao_brl"].lt(20_000)
)

casos_regra_1_n2 = resumo_regra_1_n2[
    resumo_regra_1_n2["sinalizado_regra_1"]
].copy()

casos_regra_1_n2

,cliente_id,dia,quantidade_operacoes,soma_operacoes_brl,maior_operacao_brl,sinalizado_regra_1
15,CLI-002,2026-05-01,4,64723.09,17998.60,True
28,CLI-003,2026-05-02,4,50846.72,18631.47,True
148,CLI-017,2026-03-08,4,64673.88,18761.22,True
276,CLI-029,2026-05-26,4,71297.68,19418.96,True


In [14]:
estatisticas_clientes_n2 = (
    df_nivel_2
    .groupby("cliente_id", as_index=False)
    .agg(
        quantidade_operacoes=("id", "count"),
        mediana_cliente_brl=("valor_brl", "median")
    )
)

estatisticas_clientes_n2.head()

,cliente_id,quantidade_operacoes,mediana_cliente_brl
0,CLI-001,10,1609.155
1,CLI-002,14,5448.605
2,CLI-003,14,4090.035
3,CLI-004,8,2790.230
4,CLI-005,11,2144.180


In [15]:
analise_regra_2_n2 = df_nivel_2.merge(
    estatisticas_clientes_n2,
    on="cliente_id",
    how="left"
)

analise_regra_2_n2["limite_5x_mediana_brl"] = (
    analise_regra_2_n2["mediana_cliente_brl"] * 5
)

In [16]:
analise_regra_2_n2["sinalizado_regra_2"] = (
    analise_regra_2_n2["quantidade_operacoes"].ge(4)
    & analise_regra_2_n2["valor_brl"].gt(
        analise_regra_2_n2["limite_5x_mediana_brl"]
    )
)

casos_regra_2_n2 = analise_regra_2_n2[
    analise_regra_2_n2["sinalizado_regra_2"]
].copy()

casos_regra_2_n2[
    [
        "id",
        "cliente_id",
        "data",
        "valor_brl",
        "mediana_cliente_brl",
        "limite_5x_mediana_brl"
    ]
]

,id,cliente_id,data,valor_brl,mediana_cliente_brl,limite_5x_mediana_brl
0,OP-00133,CLI-014,2026-03-06,23640.970,2308.410,11542.050
18,OP-00197,CLI-021,2026-05-12,15785.390,2832.545,14162.725
19,OP-00129,CLI-014,2026-03-05,13660.650,2308.410,11542.050
24,OP-00253,CLI-026,2026-05-17,21261.010,2032.930,10164.650
52,OP-00219,CLI-023,2026-04-28,41768.170,3241.160,16205.800
82,OP-00008,CLI-001,2026-03-15,25110.150,1609.155,8045.775
110,OP-00049,CLI-005,2026-04-12,11988.170,2144.180,10720.900
117,OP-00310,CLI-013,2026-03-06,28487.760,3146.225,15731.125
121,OP-00312,CLI-022,2026-05-27,30894.858,1909.890,9549.450
126,OP-00316,CLI-024,2026-04-24,68247.144,2740.780,13703.900


In [17]:
sinalizacoes_regra_1 = (
    casos_regra_1_n2
    .groupby("cliente_id")
    .size()
    .rename("sinalizacoes_regra_1")
)

In [20]:
sinalizacoes_regra_1

cliente_id
CLI-002    1
CLI-003    1
CLI-017    1
CLI-029    1
Name: sinalizacoes_regra_1, dtype: int64

In [18]:
sinalizacoes_regra_2 = (
    casos_regra_2_n2
    .groupby("cliente_id")
    .size()
    .rename("sinalizacoes_regra_2")
)

In [21]:
sinalizacoes_regra_2

cliente_id
CLI-001    2
CLI-005    2
CLI-007    1
CLI-008    1
CLI-013    2
CLI-014    3
CLI-021    1
CLI-022    1
CLI-023    2
CLI-024    1
CLI-026    2
CLI-028    2
CLI-030    1
Name: sinalizacoes_regra_2, dtype: int64

In [22]:
volume_clientes_n2 = (
    df_nivel_2
    .groupby("cliente_id")["valor_brl"]
    .sum()
    .rename("volume_total_brl")
)

In [23]:
ranking_clientes_n2 = pd.concat(
    [
        sinalizacoes_regra_1,
        sinalizacoes_regra_2,
        volume_clientes_n2
    ],
    axis=1
).fillna(0).reset_index()

ranking_clientes_n2[
    ["sinalizacoes_regra_1", "sinalizacoes_regra_2"]
] = ranking_clientes_n2[
    ["sinalizacoes_regra_1", "sinalizacoes_regra_2"]
].astype(int)

ranking_clientes_n2["total_sinalizacoes"] = (
    ranking_clientes_n2["sinalizacoes_regra_1"]
    + ranking_clientes_n2["sinalizacoes_regra_2"]
)

In [24]:
top_10_clientes = (
    ranking_clientes_n2[
        ranking_clientes_n2["total_sinalizacoes"] > 0
    ]
    .sort_values(
        [
            "total_sinalizacoes",
            "volume_total_brl"
        ],
        ascending=[False, False]
    )
    .head(10)
    .reset_index(drop=True)
)

top_10_clientes

,cliente_id,sinalizacoes_regra_1,sinalizacoes_regra_2,volume_total_brl,total_sinalizacoes
0,CLI-014,0,3,80629.990,3
1,CLI-023,0,2,148535.016,2
2,CLI-028,0,2,88750.800,2
3,CLI-013,0,2,81730.990,2
4,CLI-005,0,2,64742.660,2
5,CLI-026,0,2,54729.280,2
6,CLI-001,0,2,47947.810,2
7,CLI-029,1,0,191385.766,1
8,CLI-017,1,0,121391.370,1
9,CLI-030,0,1,117780.886,1


In [25]:
assert len(top_10_clientes) <= 10

assert top_10_clientes["total_sinalizacoes"].is_monotonic_decreasing

assert (
    top_10_clientes["total_sinalizacoes"]
    == (
        top_10_clientes["sinalizacoes_regra_1"]
        + top_10_clientes["sinalizacoes_regra_2"]
    )
).all()

print("Ranking validado corretamente.")

Ranking validado corretamente.


In [26]:
raiz_projeto = Path.cwd().parent

pasta_outputs = raiz_projeto / "outputs"
pasta_outputs.mkdir(exist_ok=True)

top_10_clientes.to_csv(
    pasta_outputs / "nivel_2_top_10_clientes.csv",
    index=False
)

print(
    "Ranking salvo em:",
    pasta_outputs / "nivel_2_top_10_clientes.csv"
)

Ranking salvo em: c:\Users\maria\Desktop\desafio-estagio-ia\outputs\nivel_2_top_10_clientes.csv


## Parte B — Agente com ferramentas

### Implementação e validação das ferramentas

In [1]:
import importlib
import tools

importlib.reload(tools)

from tools import (
    historico_cliente,
    operacoes_do_dia,
    perfil_canal,
    FERRAMENTAS_DISPONIVEIS
)

print(
    "Ferramentas disponíveis:",
    list(FERRAMENTAS_DISPONIVEIS)
)

Ferramentas disponíveis: ['historico_cliente', 'operacoes_do_dia', 'perfil_canal']


In [2]:
teste_historico = historico_cliente(
    "CLI-014"
)

teste_historico


{'cliente_id': 'CLI-014',
 'quantidade_operacoes': 11,
 'volume_total_brl': 80629.99,
 'valor_medio_brl': 7330.0,
 'mediana_brl': 2308.41,
 'maior_operacao_brl': 23640.97,
 'primeira_data': '2026-03-01',
 'ultima_data': '2026-05-26'}

In [4]:
datas_cliente_teste = (
    tools.df_operacoes.loc[
        (
            tools.df_operacoes["cliente_id"]
            == "CLI-014"
        )
        & tools.df_operacoes["data"].notna(),
        "data"
    ]
    .dt.strftime("%Y-%m-%d")
    .unique()
)

datas_cliente_teste

<StringArray>
['2026-03-06', '2026-05-26', '2026-03-05', '2026-03-10', '2026-03-19',
 '2026-05-11', '2026-03-03', '2026-05-22', '2026-03-24', '2026-05-10',
 '2026-03-01']
Length: 11, dtype: str

In [5]:
data_teste = datas_cliente_teste[0]

teste_operacoes_dia = operacoes_do_dia(
    cliente_id="CLI-014",
    data=data_teste
)

print("Data consultada:", data_teste)
teste_operacoes_dia

Data consultada: 2026-03-06


[{'id': 'OP-00133',
  'data': '2026-03-06',
  'valor_brl': 23640.97,
  'canal': 'pix',
  'tipo': 'pagamento',
  'contraparte': 'Mirante Transportes ME',
  'observacao': ''}]

In [7]:
teste_historico = historico_cliente(
    "CLI-014"
)

teste_perfil = perfil_canal(
    "CLI-014"
)

data_teste = datas_cliente_teste[0]

teste_operacoes_dia = operacoes_do_dia(
    cliente_id="CLI-014",
    data=data_teste
)

In [8]:
assert teste_historico["cliente_id"] == "CLI-014"

assert teste_historico["volume_total_brl"] == 80629.99

assert isinstance(teste_perfil, list)

assert isinstance(teste_operacoes_dia, list)

assert sum(
    item["quantidade_operacoes"]
    for item in teste_perfil
) == teste_historico["quantidade_operacoes"]

print("As três ferramentas foram validadas.")

As três ferramentas foram validadas.


In [9]:
import importlib
import agente

importlib.reload(agente)

from agente import executar_agente

print("Agente importado corretamente.")

Agente importado corretamente.


In [16]:
cliente_teste = "CLI-014"

df_teste = tools.df_operacoes.copy()

estatisticas_teste = (
    df_teste
    .groupby("cliente_id", as_index=False)
    .agg(
        quantidade_operacoes=("id", "count"),
        mediana_cliente_brl=("valor_brl", "median")
    )
)

analise_regra_2_teste = df_teste.merge(
    estatisticas_teste,
    on="cliente_id",
    how="left"
)

analise_regra_2_teste[
    "limite_5x_mediana_brl"
] = (
    analise_regra_2_teste[
        "mediana_cliente_brl"
    ] * 5
)

analise_regra_2_teste[
    "sinalizado_regra_2"
] = (
    analise_regra_2_teste[
        "quantidade_operacoes"
    ].ge(4)
    & analise_regra_2_teste[
        "valor_brl"
    ].gt(
        analise_regra_2_teste[
            "limite_5x_mediana_brl"
        ]
    )
)

casos_cliente_teste = (
    analise_regra_2_teste.loc[
        (
            analise_regra_2_teste["cliente_id"]
            == cliente_teste
        )
        & analise_regra_2_teste[
            "sinalizado_regra_2"
        ],
        [
            "id",
            "data",
            "valor_brl",
            "mediana_cliente_brl",
            "limite_5x_mediana_brl"
        ]
    ]
    .to_dict(orient="records")
)

alertas_cliente_teste = {
    "regra_1": [],
    "regra_2": casos_cliente_teste
}

alertas_cliente_teste

{'regra_1': [],
 'regra_2': [{'id': 'OP-00133',
   'data': Timestamp('2026-03-06 00:00:00'),
   'valor_brl': 23640.97,
   'mediana_cliente_brl': 2308.41,
   'limite_5x_mediana_brl': 11542.05},
  {'id': 'OP-00129',
   'data': Timestamp('2026-03-05 00:00:00'),
   'valor_brl': 13660.65,
   'mediana_cliente_brl': 2308.41,
   'limite_5x_mediana_brl': 11542.05},
  {'id': 'OP-00315',
   'data': Timestamp('2026-03-24 00:00:00'),
   'valor_brl': 21064.59,
   'mediana_cliente_brl': 2308.41,
   'limite_5x_mediana_brl': 11542.05}]}

In [17]:
print(
    "Alertas da Regra 1:",
    len(alertas_cliente_teste["regra_1"])
)

print(
    "Alertas da Regra 2:",
    len(alertas_cliente_teste["regra_2"])
)

Alertas da Regra 1: 0
Alertas da Regra 2: 3


In [18]:
resultado_teste_agente = executar_agente(
    cliente_id=cliente_teste,
    alertas_regras=alertas_cliente_teste
)

resultado_teste_agente

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'cliente_id': 'CLI-014',
 'modelo': 'gemini-3.5-flash-lite',
 'ferramentas_executadas': [{'nome': 'historico_cliente',
   'data': None,
   'motivo': 'Necessário para consultar o volume, média, mediana e histórico geral do cliente CLI-014 a fim de contextualizar os alertas de valor elevado.'}],
 'resultados_ferramentas': {'historico_cliente': {'cliente_id': 'CLI-014',
   'quantidade_operacoes': 11,
   'volume_total_brl': 80629.99,
   'valor_medio_brl': 7330.0,
   'mediana_brl': 2308.41,
   'maior_operacao_brl': 23640.97,
   'primeira_data': '2026-03-01',
   'ultima_data': '2026-05-26'}},
 'parecer': {'nivel_risco': 'médio',
  'tipologia_suspeita': 'Operações com valores significativamente acima da mediana histórica do cliente',
  'red_flags': ['Operação OP-00133 no valor de 23.640,97 BRL excede o limite de 5x a mediana do cliente (11.542,05 BRL).',
   'Operação OP-00129 no valor de 13.660,65 BRL excede o limite de 5x a mediana do cliente (11.542,05 BRL).',
   'Operação OP-00315 no valo

## Parte C — Execução do agente em lote

In [19]:
import pandas as pd

df_lote = tools.df_operacoes.copy()

# Regra 1
df_regra_1_lote = df_lote[
    df_lote["data"].notna()
].copy()

df_regra_1_lote["dia"] = (
    df_regra_1_lote["data"].dt.normalize()
)

resumo_regra_1_lote = (
    df_regra_1_lote
    .groupby(
        ["cliente_id", "dia"],
        as_index=False
    )
    .agg(
        quantidade_operacoes=("id", "count"),
        soma_operacoes_brl=("valor_brl", "sum"),
        maior_operacao_brl=("valor_brl", "max")
    )
)

resumo_regra_1_lote[
    "sinalizado_regra_1"
] = (
    resumo_regra_1_lote[
        "quantidade_operacoes"
    ].ge(3)
    & resumo_regra_1_lote[
        "soma_operacoes_brl"
    ].gt(50_000)
    & resumo_regra_1_lote[
        "maior_operacao_brl"
    ].lt(20_000)
)

casos_regra_1_lote = resumo_regra_1_lote[
    resumo_regra_1_lote["sinalizado_regra_1"]
].copy()

In [20]:
estatisticas_lote = (
    df_lote
    .groupby("cliente_id", as_index=False)
    .agg(
        quantidade_operacoes=("id", "count"),
        mediana_cliente_brl=("valor_brl", "median")
    )
)

analise_regra_2_lote = df_lote.merge(
    estatisticas_lote,
    on="cliente_id",
    how="left"
)

analise_regra_2_lote[
    "limite_5x_mediana_brl"
] = (
    analise_regra_2_lote[
        "mediana_cliente_brl"
    ] * 5
)

analise_regra_2_lote[
    "sinalizado_regra_2"
] = (
    analise_regra_2_lote[
        "quantidade_operacoes"
    ].ge(4)
    & analise_regra_2_lote[
        "valor_brl"
    ].gt(
        analise_regra_2_lote[
            "limite_5x_mediana_brl"
        ]
    )
)

casos_regra_2_lote = analise_regra_2_lote[
    analise_regra_2_lote[
        "sinalizado_regra_2"
    ]
].copy()

In [21]:
contagem_r1 = (
    casos_regra_1_lote
    .groupby("cliente_id")
    .size()
    .rename("sinalizacoes_regra_1")
)

contagem_r2 = (
    casos_regra_2_lote
    .groupby("cliente_id")
    .size()
    .rename("sinalizacoes_regra_2")
)

volume_lote = (
    df_lote
    .groupby("cliente_id")["valor_brl"]
    .sum()
    .rename("volume_total_brl")
)

ranking_lote = pd.concat(
    [
        contagem_r1,
        contagem_r2,
        volume_lote
    ],
    axis=1
).fillna(0).reset_index()

ranking_lote[
    ["sinalizacoes_regra_1", "sinalizacoes_regra_2"]
] = ranking_lote[
    ["sinalizacoes_regra_1", "sinalizacoes_regra_2"]
].astype(int)

ranking_lote["total_sinalizacoes"] = (
    ranking_lote["sinalizacoes_regra_1"]
    + ranking_lote["sinalizacoes_regra_2"]
)

top_10_lote = (
    ranking_lote[
        ranking_lote["total_sinalizacoes"] > 0
    ]
    .sort_values(
        ["total_sinalizacoes", "volume_total_brl"],
        ascending=[False, False]
    )
    .head(10)
    .reset_index(drop=True)
)

top_10_lote

,cliente_id,sinalizacoes_regra_1,sinalizacoes_regra_2,volume_total_brl,total_sinalizacoes
0,CLI-014,0,3,80629.990,3
1,CLI-023,0,2,148535.016,2
2,CLI-028,0,2,88750.800,2
3,CLI-013,0,2,81730.990,2
4,CLI-005,0,2,64742.660,2
5,CLI-026,0,2,54729.280,2
6,CLI-001,0,2,47947.810,2
7,CLI-029,1,0,191385.766,1
8,CLI-017,1,0,121391.370,1
9,CLI-030,0,1,117780.886,1


In [22]:
def montar_alertas_cliente(cliente_id):
    alertas_regra_1 = (
        casos_regra_1_lote.loc[
            casos_regra_1_lote["cliente_id"]
            == cliente_id,
            [
                "dia",
                "quantidade_operacoes",
                "soma_operacoes_brl",
                "maior_operacao_brl"
            ]
        ]
        .copy()
    )

    alertas_regra_1["dia"] = (
        alertas_regra_1["dia"]
        .dt.strftime("%Y-%m-%d")
    )

    alertas_regra_2 = (
        casos_regra_2_lote.loc[
            casos_regra_2_lote["cliente_id"]
            == cliente_id,
            [
                "id",
                "data",
                "valor_brl",
                "mediana_cliente_brl",
                "limite_5x_mediana_brl"
            ]
        ]
        .copy()
    )

    alertas_regra_2["data"] = (
        alertas_regra_2["data"]
        .dt.strftime("%Y-%m-%d")
    )

    return {
        "regra_1": alertas_regra_1.to_dict(
            orient="records"
        ),
        "regra_2": alertas_regra_2.to_dict(
            orient="records"
        )
    }

In [23]:
import json
from pathlib import Path

raiz_projeto = Path.cwd().parent

pasta_outputs = raiz_projeto / "outputs"
pasta_outputs.mkdir(exist_ok=True)

print("Resultados serão salvos em:", pasta_outputs)

Resultados serão salvos em: c:\Users\maria\Desktop\desafio-estagio-ia\outputs


In [24]:
resultados_lote = []
erros_lote = []

for posicao, linha in top_10_lote.iterrows():
    cliente_id = linha["cliente_id"]

    print(
        f"[{posicao + 1}/10] Analisando {cliente_id}..."
    )

    try:
        alertas = montar_alertas_cliente(
            cliente_id
        )

        resultado = executar_agente(
            cliente_id=cliente_id,
            alertas_regras=alertas
        )

        resultado["regras_deterministicas"] = {
            "sinalizacoes_regra_1": int(
                linha["sinalizacoes_regra_1"]
            ),
            "sinalizacoes_regra_2": int(
                linha["sinalizacoes_regra_2"]
            ),
            "total_sinalizacoes": int(
                linha["total_sinalizacoes"]
            ),
            "volume_total_brl": float(
                linha["volume_total_brl"]
            )
        }

        resultados_lote.append(resultado)

        caminho_cliente = (
            pasta_outputs
            / f"nivel_2_{cliente_id}.json"
        )

        with open(
            caminho_cliente,
            "w",
            encoding="utf-8"
        ) as arquivo:
            json.dump(
                resultado,
                arquivo,
                ensure_ascii=False,
                indent=2,
                default=str
            )

        print(f"{cliente_id}: concluído.")

    except Exception as erro:
        erros_lote.append({
            "cliente_id": cliente_id,
            "erro": str(erro)
        })

        print(
            f"{cliente_id}: erro — {erro}"
        )

[1/10] Analisando CLI-014...
CLI-014: concluído.
[2/10] Analisando CLI-023...
CLI-023: concluído.
[3/10] Analisando CLI-028...
CLI-028: concluído.
[4/10] Analisando CLI-013...
CLI-013: concluído.
[5/10] Analisando CLI-005...
CLI-005: concluído.
[6/10] Analisando CLI-026...
CLI-026: concluído.
[7/10] Analisando CLI-001...
CLI-001: concluído.
[8/10] Analisando CLI-029...
CLI-029: concluído.
[9/10] Analisando CLI-017...
CLI-017: concluído.
[10/10] Analisando CLI-030...
CLI-030: concluído.


In [25]:
registros_metricas = []

for resultado in resultados_lote:
    for chamada in resultado["metricas_chamadas"]:
        registros_metricas.append({
            "cliente_id": resultado["cliente_id"],
            "modelo": resultado["modelo"],
            "etapa": chamada["etapa"],
            "latencia_segundos": (
                chamada["latencia_segundos"]
            ),
            "tokens_entrada": (
                chamada["tokens_entrada"]
            ),
            "tokens_saida": (
                chamada["tokens_saida"]
            ),
            "tokens_total": (
                chamada["tokens_total"]
            )
        })

df_metricas = pd.DataFrame(
    registros_metricas
)

df_metricas


,cliente_id,modelo,etapa,latencia_segundos,tokens_entrada,tokens_saida,tokens_total
0,CLI-014,gemini-3.5-flash-lite,planejamento,187.139,523,124,647
1,CLI-014,gemini-3.5-flash-lite,parecer,296.357,996,386,1382
2,CLI-023,gemini-3.5-flash-lite,planejamento,1.250,431,201,632
3,CLI-023,gemini-3.5-flash-lite,parecer,39.593,762,320,1082
4,CLI-028,gemini-3.5-flash-lite,planejamento,1.345,424,85,509
5,CLI-028,gemini-3.5-flash-lite,parecer,1.922,696,373,1069
6,CLI-013,gemini-3.5-flash-lite,planejamento,54.741,428,150,578
7,CLI-013,gemini-3.5-flash-lite,parecer,198.302,848,339,1187
8,CLI-005,gemini-3.5-flash-lite,planejamento,1.550,422,247,669
9,CLI-005,gemini-3.5-flash-lite,parecer,50.621,753,298,1051


In [26]:
assert len(df_metricas) == 20

assert df_metricas[
    "cliente_id"
].nunique() == 10

assert set(
    df_metricas["etapa"]
) == {
    "planejamento",
    "parecer"
}

print("Métricas validadas.")

Métricas validadas.


In [27]:
CAMADA_GRATUITA = True

if CAMADA_GRATUITA:
    df_metricas["custo_usd"] = 0.0
    df_metricas["tipo_custo"] = (
        "Camada gratuita"
    )
else:
    raise ValueError(
        "Informe os preços oficiais do modelo "
        "antes de calcular o custo."
    )

df_metricas.head()

,cliente_id,modelo,etapa,latencia_segundos,tokens_entrada,tokens_saida,tokens_total,custo_usd,tipo_custo
0,CLI-014,gemini-3.5-flash-lite,planejamento,187.139,523,124,647,0.0,Camada gratuita
1,CLI-014,gemini-3.5-flash-lite,parecer,296.357,996,386,1382,0.0,Camada gratuita
2,CLI-023,gemini-3.5-flash-lite,planejamento,1.250,431,201,632,0.0,Camada gratuita
3,CLI-023,gemini-3.5-flash-lite,parecer,39.593,762,320,1082,0.0,Camada gratuita
4,CLI-028,gemini-3.5-flash-lite,planejamento,1.345,424,85,509,0.0,Camada gratuita


In [28]:
resumo_metricas = pd.DataFrame({
    "quantidade_clientes": [
        df_metricas["cliente_id"].nunique()
    ],
    "quantidade_chamadas": [
        len(df_metricas)
    ],
    "latencia_total_segundos": [
        df_metricas[
            "latencia_segundos"
        ].sum()
    ],
    "latencia_media_segundos": [
        df_metricas[
            "latencia_segundos"
        ].mean()
    ],
    "tokens_entrada_total": [
        df_metricas[
            "tokens_entrada"
        ].sum()
    ],
    "tokens_saida_total": [
        df_metricas[
            "tokens_saida"
        ].sum()
    ],
    "tokens_total": [
        df_metricas[
            "tokens_total"
        ].sum()
    ],
    "custo_total_usd": [
        df_metricas[
            "custo_usd"
        ].sum()
    ]
})

resumo_metricas.round(3)

,quantidade_clientes,quantidade_chamadas,latencia_total_segundos,latencia_media_segundos,tokens_entrada_total,tokens_saida_total,tokens_total,custo_total_usd
0,10,20,1092.239,54.612,12143,4777,16920,0.0


In [29]:
metricas_por_etapa = (
    df_metricas
    .groupby("etapa", as_index=False)
    .agg(
        quantidade_chamadas=(
            "cliente_id",
            "count"
        ),
        latencia_media_segundos=(
            "latencia_segundos",
            "mean"
        ),
        tokens_entrada=(
            "tokens_entrada",
            "sum"
        ),
        tokens_saida=(
            "tokens_saida",
            "sum"
        ),
        tokens_total=(
            "tokens_total",
            "sum"
        ),
        custo_total_usd=(
            "custo_usd",
            "sum"
        )
    )
)

metricas_por_etapa.round(3)

,etapa,quantidade_chamadas,latencia_media_segundos,tokens_entrada,tokens_saida,tokens_total,custo_total_usd
0,parecer,10,71.068,8135,3216,11351,0.0
1,planejamento,10,38.156,4008,1561,5569,0.0


In [30]:
df_metricas.to_csv(
    pasta_outputs
    / "nivel_2_metricas_chamadas.csv",
    index=False
)

resumo_metricas.to_csv(
    pasta_outputs
    / "nivel_2_resumo_metricas.csv",
    index=False
)

metricas_por_etapa.to_csv(
    pasta_outputs
    / "nivel_2_metricas_por_etapa.csv",
    index=False
)

print("Métricas salvas.")

Métricas salvas.


In [31]:
import importlib
import confronto

importlib.reload(confronto)

from confronto import gerar_confronto

df_confronto, resumo_confronto = (
    gerar_confronto()
)

resumo_confronto

{'criterio': 'Risco alto quando as duas regras sinalizam; médio quando apenas uma regra sinaliza; baixo quando nenhuma regra sinaliza.',
 'quantidade_clientes': 10,
 'quantidade_concordancias': 8,
 'quantidade_divergencias': 2,
 'taxa_concordancia': 0.8}

In [32]:
df_confronto[
    [
        "cliente_id",
        "sinalizacoes_regra_1",
        "sinalizacoes_regra_2",
        "risco_deterministico",
        "risco_agente",
        "concorda"
    ]
].sort_values(
    "cliente_id"
)

,cliente_id,sinalizacoes_regra_1,sinalizacoes_regra_2,risco_deterministico,risco_agente,concorda
0,CLI-001,0,2,médio,médio,True
1,CLI-005,0,2,médio,médio,True
2,CLI-013,0,2,médio,médio,True
3,CLI-014,0,3,médio,médio,True
4,CLI-017,1,0,médio,baixo,False
5,CLI-023,0,2,médio,médio,True
6,CLI-026,0,2,médio,médio,True
7,CLI-028,0,2,médio,médio,True
8,CLI-029,1,0,médio,baixo,False
9,CLI-030,0,1,médio,médio,True


In [33]:
divergencias = df_confronto[
    ~df_confronto["concorda"]
].copy()

divergencias[
    [
        "cliente_id",
        "risco_deterministico",
        "risco_agente",
        "justificativa_agente"
    ]
]

,cliente_id,risco_deterministico,risco_agente,justificativa_agente
4,CLI-017,médio,baixo,O alerta determinístico da regra_1 foi dispara...
8,CLI-029,médio,baixo,Parecer preliminar baseado exclusivamente nos ...


In [34]:
pd.set_option(
    "display.max_colwidth",
    None
)

divergencias[
    [
        "cliente_id",
        "sinalizacoes_regra_1",
        "sinalizacoes_regra_2",
        "risco_deterministico",
        "risco_agente",
        "justificativa_agente"
    ]
]

,cliente_id,sinalizacoes_regra_1,sinalizacoes_regra_2,risco_deterministico,risco_agente,justificativa_agente
4,CLI-017,1,0,médio,baixo,"O alerta determinístico da regra_1 foi disparado devido ao volume e quantidade de operações concentradas na data de 2026-03-08. No entanto, os resultados das ferramentas mostram que o cliente possui um histórico total de 13 operações distribuídas até a data de 2026-05-24, com valor médio de R$ 9.337,80 e maior operação de R$ 18.761,22, patamares compatíveis com os montantes movimentados no dia do alerta. Destaca-se que regras determinísticas podem gerar falsos positivos e uma única sinalização não comprova irregularidade. Há limitações na análise devido à ausência de dados cadastrais, perfil socioeconômico e finalidade declarada das transações."
8,CLI-029,1,0,médio,baixo,"Parecer preliminar baseado exclusivamente nos dados fornecidos. O cliente CLI-029 possui histórico de 16 operações totalizando R$ 191.385,77 desde 2026-03-11, com valor médio de R$ 11.961,61. O volume e os valores operados em 2026-05-26 alinham-se à magnitude geral do histórico do cliente. As regras determinísticas podem gerar falsos positivos e uma sinalização isolada não comprova irregularidade, havendo limitação de dados por ausência de informações qualitativas sobre a motivação comercial das contrapartes (Orion Trading SA, Solar Importacao LTDA, Rubi Importacao ME e Cristal Atacado ME)."


In [35]:
with open(
    pasta_outputs
    / "nivel_2_resultados_lote.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        resultados_lote,
        arquivo,
        ensure_ascii=False,
        indent=2,
        default=str
    )

print("Resultados consolidados salvos.")

Resultados consolidados salvos.


In [36]:
arquivos_clientes = list(
    pasta_outputs.glob(
        "nivel_2_CLI-*.json"
    )
)

assert len(arquivos_clientes) == 10

assert len(resultados_lote) == 10

assert len(df_metricas) == 20

assert len(df_confronto) == 10

assert (
    pasta_outputs
    / "nivel_2_resultados_lote.json"
).exists()

assert (
    pasta_outputs
    / "nivel_2_confronto.csv"
).exists()

assert (
    pasta_outputs
    / "nivel_2_confronto_resumo.json"
).exists()

print("Nível 2 validado.")

Nível 2 validado.


In [37]:
analises_divergencias = {
    "CLI-017": {
        "quem_parece_correto": "regra determinística",
        "analise_divergencia": (
            "A regra determinística parece mais adequada. "
            "O cliente apresentou operações concentradas no mesmo "
            "dia que atenderam integralmente aos critérios da Regra 1. "
            "A compatibilidade dos valores individuais com o histórico "
            "não elimina o padrão temporal observado. Ainda assim, a "
            "sinalização representa um alerta preliminar e não comprova "
            "irregularidade."
        )
    },
    "CLI-029": {
        "quem_parece_correto": "agente",
        "analise_divergencia": (
            "O agente parece mais adequado. Embora a concentração das "
            "operações tenha acionado formalmente a Regra 1, os valores "
            "do dia estavam alinhados à magnitude do histórico do "
            "cliente, que possui 16 operações e volume total próximo "
            "de R$ 191 mil. Isso oferece uma justificativa plausível "
            "para tratar o caso como possível falso positivo, sem "
            "dispensar revisão humana."
        )
    }
}

df_confronto["quem_parece_correto"] = (
    df_confronto["cliente_id"]
    .map(
        {
            cliente: analise[
                "quem_parece_correto"
            ]
            for cliente, analise
            in analises_divergencias.items()
        }
    )
    .fillna("regra e agente concordaram")
)

df_confronto["analise_divergencia"] = (
    df_confronto["cliente_id"]
    .map(
        {
            cliente: analise[
                "analise_divergencia"
            ]
            for cliente, analise
            in analises_divergencias.items()
        }
    )
    .fillna(
        "Não houve divergência entre a classificação "
        "determinística e a classificação do agente."
    )
)

df_confronto[
    [
        "cliente_id",
        "risco_deterministico",
        "risco_agente",
        "concorda",
        "quem_parece_correto",
        "analise_divergencia"
    ]
]

,cliente_id,risco_deterministico,risco_agente,concorda,quem_parece_correto,analise_divergencia
0,CLI-001,médio,médio,True,regra e agente concordaram,Não houve divergência entre a classificação determinística e a classificação do agente.
1,CLI-005,médio,médio,True,regra e agente concordaram,Não houve divergência entre a classificação determinística e a classificação do agente.
2,CLI-013,médio,médio,True,regra e agente concordaram,Não houve divergência entre a classificação determinística e a classificação do agente.
3,CLI-014,médio,médio,True,regra e agente concordaram,Não houve divergência entre a classificação determinística e a classificação do agente.
4,CLI-017,médio,baixo,False,regra determinística,"A regra determinística parece mais adequada. O cliente apresentou operações concentradas no mesmo dia que atenderam integralmente aos critérios da Regra 1. A compatibilidade dos valores individuais com o histórico não elimina o padrão temporal observado. Ainda assim, a sinalização representa um alerta preliminar e não comprova irregularidade."
5,CLI-023,médio,médio,True,regra e agente concordaram,Não houve divergência entre a classificação determinística e a classificação do agente.
6,CLI-026,médio,médio,True,regra e agente concordaram,Não houve divergência entre a classificação determinística e a classificação do agente.
7,CLI-028,médio,médio,True,regra e agente concordaram,Não houve divergência entre a classificação determinística e a classificação do agente.
8,CLI-029,médio,baixo,False,agente,"O agente parece mais adequado. Embora a concentração das operações tenha acionado formalmente a Regra 1, os valores do dia estavam alinhados à magnitude do histórico do cliente, que possui 16 operações e volume total próximo de R$ 191 mil. Isso oferece uma justificativa plausível para tratar o caso como possível falso positivo, sem dispensar revisão humana."
9,CLI-030,médio,médio,True,regra e agente concordaram,Não houve divergência entre a classificação determinística e a classificação do agente.


In [ ]:
df_confronto.to_csv(
    pasta_outputs
    / "nivel_2_confronto.csv",
    index=False
)
resultado_confronto_completo = {
    "criterio_correspondencia": (
        "Risco alto quando as duas regras sinalizam; "
        "médio quando apenas uma regra sinaliza; "
        "baixo quando nenhuma regra sinaliza."
    ),
    "quantidade_clientes": 10,
    "quantidade_concordancias": 8,
    "quantidade_divergencias": 2,
    "taxa_concordancia": 0.8,
    "analise_divergencias": (
        analises_divergencias
    )
}

with open(
    pasta_outputs
    / "nivel_2_confronto_resumo.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        resultado_confronto_completo,
        arquivo,
        ensure_ascii=False,
        indent=2
    )

print(
    "Confronto e divergências salvos."
)

Confronto e divergências salvos.


In [39]:
assert len(df_confronto) == 10

assert df_confronto["concorda"].sum() == 8

assert (~df_confronto["concorda"]).sum() == 2

assert round(
    df_confronto["concorda"].mean(),
    2
) == 0.80

assert df_confronto[
    "analise_divergencia"
].notna().all()

print(
    "Nível 2 concluído e validado."
)

Nível 2 concluído e validado.
